# Feature EDA & QA

This notebook audits rolling features (R3, R5) and checks for leakage, missing data, and calibration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False

# Replace this with your DataFrame source
# Example: df = pd.read_parquet('data/processed/features.parquet')
df = df  # noqa: F821  # expects df to already exist in the notebook context

sns.set_style('whitegrid')

## Missing Value Audit

In [ ]:
feature_cols = [c for c in df.columns if c.startswith(('OFF_', 'DEF_', 'DIS_', 'CTX_', 'MKT_'))]
target_col = 'home_win'

nan_pct = df[feature_cols].isna().mean().sort_values(ascending=False) * 100
display(nan_pct.to_frame('nan_pct'))

plt.figure(figsize=(12, 4))
nan_pct.plot(kind='bar')
plt.title('NaN Percentage per Feature')
plt.ylabel('% NaN')
plt.tight_layout()
plt.show()

if HAS_MISSINGNO:
    msno.bar(df[feature_cols], figsize=(12, 4))
    plt.show()

## Distribution Analysis (OFF_/DEF_)

In [ ]:
off_def_cols = [c for c in feature_cols if c.startswith(('OFF_', 'DEF_'))]

for col in off_def_cols:
    plt.figure(figsize=(6, 3))
    sns.histplot(df[col].dropna(), kde=True, bins=30)
    plt.title(f'Distribution: {col}')
    plt.tight_layout()
    plt.show()

## Leakage & Correlation Check

In [ ]:
corr = df[feature_cols + [target_col]].corr(numeric_only=True)[target_col].sort_values(ascending=False)
display(corr.to_frame('corr_with_target'))

plt.figure(figsize=(8, 10))
sns.heatmap(df[feature_cols + [target_col]].corr(numeric_only=True)[[target_col]], annot=False, cmap='coolwarm')
plt.title('Correlation with home_win')
plt.tight_layout()
plt.show()

leak_flags = corr[abs(corr) > 0.8]
if not leak_flags.empty:
    print('Potential leakage detected (|corr| > 0.8):')
    display(leak_flags)
else:
    print('No high-correlation leakage flags detected.')

## Feature vs Target Boxplots

In [ ]:
top5 = corr.drop(target_col).head(5).index.tolist()
for col in top5:
    plt.figure(figsize=(5, 3))
    sns.boxplot(x=target_col, y=col, data=df)
    plt.title(f'{col} vs {target_col}')
    plt.tight_layout()
    plt.show()

## Time-Series Consistency

In [ ]:
date_col = 'date' if 'date' in df.columns else 'Date'
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
ts_col = next((c for c in df.columns if c.startswith('OFF_') and c.endswith('_R5') and 'FTHG' in c), None)
if ts_col:
    ts = df[[date_col, ts_col]].dropna().sort_values(date_col)
    ts_grouped = ts.groupby(pd.Grouper(key=date_col, freq='W'))[ts_col].mean()
    plt.figure(figsize=(10, 4))
    ts_grouped.plot()
    plt.title(f'Weekly Average of {ts_col}')
    plt.tight_layout()
    plt.show()
else:
    print('No OFF_*FTHG*_R5 feature found for time-series plot.')

## Market Efficiency Check

In [ ]:
prob_col = next((c for c in df.columns if c.startswith('MKT_') and 'HOME' in c), None)
if prob_col:
    tmp = df[[prob_col, target_col]].dropna()
    tmp['bin'] = pd.cut(tmp[prob_col], bins=10)
    calib = tmp.groupby('bin')[target_col].mean()
    plt.figure(figsize=(6, 4))
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.plot(calib.index.map(lambda x: x.mid), calib.values, marker='o')
    plt.title('Market Implied Home Prob vs Actual')
    plt.xlabel('Implied Probability (bin mid)')
    plt.ylabel('Actual Home Win Rate')
    plt.tight_layout()
    plt.show()
else:
    print('No MKT_*HOME* feature found for calibration plot.')